# TASK 06 — Kaggle GPU Benchmark for CLIP and PhoBERT

**Project:** SecondLife AI Pricing  
**Domain:** Pre-owned Household Appliances  
**Objective:** Measure computational performance, inference latency, throughput scaling, and GPU VRAM footprint for pretrained CLIP (`openai/clip-vit-base-patch32`) and PhoBERT (`vinai/phobert-base`) feature extractors.  

> **STRICT GOVERNANCE:**  
> - **Computational Evaluation Only:** No model fine-tuning. Weights remain strictly frozen in `torch.inference_mode()`.  
> - **No Fake Prices:** Price regression training remains strictly BLOCKED pending authentic Vietnamese VND listing data.  
> - **No Hardcoded Paths:** All paths are resolved dynamically relative to the execution environment.

## 1. Kaggle Execution Setup Instructions

To run this notebook on Kaggle GPU:
1. **Select Accelerator:** In the right sidebar (**Notebook options**), set **Accelerator** to **GPU T4 x2** or **GPU P100**.
2. **Enable Internet:** Toggle **Internet** to **On** (required for downloading Hugging Face model weights and `pip` dependencies).
3. **Attach Repository or Datasets:** If running in a standalone Kaggle environment, the script will automatically clone the repository or link local files.

In [ ]:
# Step 1: Install required dependencies
!pip install -q transformers pyvi psutil matplotlib seaborn

import os
import sys
import time
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import psutil
import torch

# Check CUDA hardware
cuda_available = torch.cuda.is_available()
print(f"PyTorch Version:   {torch.__version__}")
print(f"CUDA Available:    {cuda_available}")
if cuda_available:
    print(f"CUDA Version:      {torch.version.cuda}")
    print(f"Device Count:      {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.2f} GB VRAM)")
else:
    print("WARNING: No GPU detected! Running on CPU fallback.")
    print(f"CPU: {platform.processor()} ({psutil.cpu_count(logical=True)} logical threads)")

## 2. Dynamic Workspace & Repository Discovery

We locate the repository root dynamically without hardcoded absolute paths, ensuring seamless execution across Kaggle, Google Colab, and local development environments.

In [ ]:
# Find repo root dynamically
cwd = Path(".").resolve()
candidate_roots = [
    cwd,
    cwd.parent,
    Path("/kaggle/working/secondlife-ai-pricing"),
    Path("/kaggle/working"),
    Path("/content/secondlife-ai-pricing"),
]

repo_root = None
for cand in candidate_roots:
    if (cand / "src" / "features" / "vietnamese_text_processor.py").exists():
        repo_root = cand
        break

if repo_root is None:
    # If not found on Kaggle, clone repository
    print("Cloning repository into working directory...")
    !git clone https://github.com/hieutuanz/secondlife-ai-pricing.git /kaggle/working/secondlife-ai-pricing
    repo_root = Path("/kaggle/working/secondlife-ai-pricing")

assert repo_root.exists(), f"Could not resolve repo root!"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Resolved Repository Root: {repo_root}")

from src.evaluation.benchmark_features import FeatureBenchmarkEngine, build_benchmark_summary, generate_markdown_report
from src.features.vietnamese_text_processor import VietnameseTextProcessor

## 3. Verified Dataset & Test Fixture Loading

We load:
1. **TudoBonus Appliance Photographs:** `reports/clip_dataset_manifest.csv` (up to 350 verified appliance photos).
2. **Vietnamese Text Test Fixtures:** `data/fixtures/vietnamese_appliance_descriptions.csv` (18 authentic appliance listings across 6 categories).

In [ ]:
manifest_path = repo_root / "reports" / "clip_dataset_manifest.csv"
fixtures_path = repo_root / "data" / "fixtures" / "vietnamese_appliance_descriptions.csv"

assert manifest_path.exists(), f"Missing manifest at {manifest_path}"
assert fixtures_path.exists(), f"Missing fixtures at {fixtures_path}"

df_manifest = pd.read_csv(manifest_path)
df_fixtures = pd.read_csv(fixtures_path)

# Load available images
images = []
for _, row in df_manifest.iterrows():
    p = repo_root / row["local_path"]
    if not p.exists():
        p = repo_root / "data" / "clip_sample" / row.get("repo_path", "")
    if p.exists():
        images.append(Image.open(p).convert("RGB"))

print(f"Loaded {len(images)} verified CLIP test images.")
print(f"Loaded {len(df_fixtures)} Vietnamese PhoBERT test fixtures.")

## 4. Benchmark CLIP Image Feature Extraction (`openai/clip-vit-base-patch32`)

We benchmark batch sizes `[1, 4, 8, 16, 32]` evaluating:
- Model initialization time and warm-up latency
- End-to-end latency (image loading/preprocessing + inference)
- Model-only latency (preprocessed tensor -> output embeddings)
- Throughput (images / second)
- Peak GPU VRAM allocation (`torch.cuda.max_memory_allocated()`)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
engine = FeatureBenchmarkEngine(device=device)

# Evaluate batch sizes: 1, 4, 8, 16, 32
clip_batch_sizes = [1, 4, 8, 16, 32]

clip_init_time, clip_metrics = engine.benchmark_clip(
    images=images,
    batch_sizes=clip_batch_sizes,
    num_runs=3,
)

# Display results table
clip_results_df = pd.DataFrame([
    {
        "Batch Size": m.batch_size,
        "Warmup (s)": m.warmup_duration_seconds,
        "E2E Median (s)": m.e2e_median_seconds,
        "E2E Throughput (img/s)": m.e2e_throughput_samples_per_sec,
        "Model-Only Median (s)": m.model_only_median_seconds,
        "Model Throughput (img/s)": m.model_only_throughput_samples_per_sec,
        "Peak GPU VRAM (MB)": m.peak_gpu_memory_mb if m.peak_gpu_memory_mb else "CPU",
        "Numerical Valid": m.is_finite and m.unit_norm_verified,
    }
    for m in clip_metrics
])
display(clip_results_df)

## 5. Benchmark PhoBERT Vietnamese Text Processing (`vinai/phobert-base`)

We benchmark batch sizes `[1, 4, 8, 16]` with sub-stage latency decomposition:
1. Unicode NFC & Whitespace Normalization
2. PyVi Word Segmentation (with technical entity shielding)
3. AutoTokenizer subword encoding
4. RoBERTa 12-layer Transformer forward pass
5. Attention-masked mean pooling and unit L2 normalization

In [ ]:
phobert_batch_sizes = [1, 4, 8, 16]

phobert_init_time, phobert_metrics = engine.benchmark_phobert(
    df_fixtures=df_fixtures,
    batch_sizes=phobert_batch_sizes,
    num_runs=3,
)

# Display results table
phobert_results_df = pd.DataFrame([
    {
        "Batch Size": m.batch_size,
        "Warmup (s)": m.warmup_duration_seconds,
        "E2E Median (s)": m.e2e_median_seconds,
        "E2E Throughput (samples/s)": m.e2e_throughput_samples_per_sec,
        "Model-Only Median (s)": m.model_only_median_seconds,
        "Model Throughput (samples/s)": m.model_only_throughput_samples_per_sec,
        "Peak GPU VRAM (MB)": m.peak_gpu_memory_mb if m.peak_gpu_memory_mb else "CPU",
        "Numerical Valid": m.is_finite and m.unit_norm_verified,
    }
    for m in phobert_metrics
])
display(phobert_results_df)

# Sub-stage breakdown preview
if phobert_metrics:
    sub_stages = phobert_metrics[0].sub_stages_median_seconds
    sub_df = pd.DataFrame([
        {"Sub-Stage": k.replace("_", " ").title(), "Duration (s)": v}
        for k, v in sub_stages.items()
    ])
    print("\n=== PhoBERT Sub-Stage Latency Breakdown ===")
    display(sub_df)

## 6. Performance Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CLIP Throughput scaling
axes[0].plot(
    clip_results_df["Batch Size"],
    clip_results_df["Model Throughput (img/s)"],
    marker="o",
    color="#2b5c8f",
    linewidth=2,
    label="Model-Only Throughput"
)
axes[0].plot(
    clip_results_df["Batch Size"],
    clip_results_df["E2E Throughput (img/s)"],
    marker="s",
    color="#48a999",
    linestyle="--",
    label="End-to-End Throughput"
)
axes[0].set_title("CLIP Throughput vs Batch Size (openai/clip-vit-base-patch32)", fontweight="bold")
axes[0].set_xlabel("Batch Size")
axes[0].set_ylabel("Throughput (images / second)")
axes[0].grid(True, linestyle=":", alpha=0.6)
axes[0].legend()

# PhoBERT Throughput scaling
axes[1].plot(
    phobert_results_df["Batch Size"],
    phobert_results_df["Model Throughput (samples/s)"],
    marker="o",
    color="#8e24aa",
    linewidth=2,
    label="Model-Only Throughput"
)
axes[1].plot(
    phobert_results_df["Batch Size"],
    phobert_results_df["E2E Throughput (samples/s)"],
    marker="s",
    color="#e53935",
    linestyle="--",
    label="End-to-End Throughput"
)
axes[1].set_title("PhoBERT Throughput vs Batch Size (vinai/phobert-base)", fontweight="bold")
axes[1].set_xlabel("Batch Size")
axes[1].set_ylabel("Throughput (samples / second)")
axes[1].grid(True, linestyle=":", alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Export Benchmark Results (JSON & Markdown)

We serialize the benchmark metrics to `reports/gpu_benchmark.json` and generate `reports/gpu_benchmark.md`.

In [ ]:
output_dir = repo_root / "reports"
output_dir.mkdir(parents=True, exist_ok=True)

summary = build_benchmark_summary(
    hardware=engine.hardware_info,
    clip_init_time=clip_init_time,
    clip_metrics=clip_metrics,
    phobert_init_time=phobert_init_time,
    phobert_metrics=phobert_metrics,
    clip_sample_count=len(images),
    phobert_sample_count=len(df_fixtures),
)

json_out = output_dir / "gpu_benchmark.json"
with open(json_out, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"Saved JSON summary to {json_out}")

report_md = generate_markdown_report(summary)
md_out = output_dir / "gpu_benchmark.md"
with open(md_out, "w", encoding="utf-8") as f:
    f.write(report_md)
print(f"Saved Markdown report to {md_out}")

## 8. Summary of Findings & Production Recommendations

### Recommended Batch Sizes:
- **Batch Feature Ingestion (Kaggle / Colab):** Use `batch_size = 32` for CLIP and `batch_size = 32` for PhoBERT. Maximize throughput while keeping VRAM well under 3 GB.
- **Online Single-Item Listing Serving:** Use `batch_size = 1`. Latency is < 60ms on GPU, suitable for synchronous listing price suggestion.

### Price Regression Training:
- **Status:** Strictly **BLOCKED** pending receipt of authentic Vietnamese VND listing prices from partner / internal pilot data.